In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.0 MB/s eta 0:00:00


In [27]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn


DAGSHUB_MLFLOW_URI = "https://dagshub.com/mkekn23/ML-assignment-1.mlflow"
mlflow.set_tracking_uri(DAGSHUB_MLFLOW_URI)

model_name = "House_Prices_Best_Model" 
model_version = "2"


model_uri = f"models:/{model_name}/{model_version}"
print(f"მოდელის გადმოწერა მიმდინარეობს: {model_uri}")

loaded_model = mlflow.sklearn.load_model(model_uri)
print("მოდელი წარმატებით ჩაიტვირთა!")


test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
test_ids = test_df['Id'] 


მოდელის გადმოწერა მიმდინარეობს: models:/House_Prices_Best_Model/2


მოდელი წარმატებით ჩაიტვირთა!


In [31]:
import pandas as pd
import numpy as np

X_test_processed = test_df.copy() 


collumns_to_fill_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtFinType2', 
    'BsmtExposure', 'BsmtCond', 'BsmtFinType1', 'BsmtQual'
]

for col in collumns_to_fill_none:
    X_test_processed[col] = X_test_processed[col].fillna("None")

X_test_processed['GarageYrBlt'] = X_test_processed['GarageYrBlt'].fillna(0)
X_test_processed['MasVnrArea'] = X_test_processed['MasVnrArea'].fillna(0)

median = X_test_processed['LotFrontage'].median()
X_test_processed['LotFrontage'] = X_test_processed['LotFrontage'].fillna(median)

X_test_processed['Electrical'] = X_test_processed['Electrical'].fillna(X_test_processed['Electrical'].mode()[0])
X_test_processed['GrLivArea'] = X_test_processed['GrLivArea'].clip(upper=4000)


for col in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']:
    X_test_processed[col] = X_test_processed[col].fillna(0)

X_test_processed['TotalSF'] = X_test_processed['TotalBsmtSF'] + X_test_processed['1stFlrSF'] + X_test_processed['2ndFlrSF']
X_test_processed['RemodAge'] = X_test_processed['YrSold'] - X_test_processed['YearRemodAdd']
X_test_processed['TotalBath'] = (X_test_processed['FullBath'] + (0.5 * X_test_processed['HalfBath']) +
                                 X_test_processed['BsmtFullBath'] + (0.5 * X_test_processed['BsmtHalfBath']))
X_test_processed['HouseAge'] = X_test_processed['YrSold'] - X_test_processed['YearBuilt']


# Label Encoding
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'Na': 0, 'None': 0}
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
                'HeatingQC', 'KitchenQual', 'FireplaceQu', 
                'GarageQual', 'GarageCond', 'PoolQC']

for col in ordinal_cols:
    X_test_processed[col] = X_test_processed[col].map(quality_map)


X_test_processed = pd.get_dummies(X_test_processed)



expected_features = loaded_model.feature_names_in_
X_test_processed = X_test_processed.reindex(columns=expected_features, fill_value=0)



num_features = X_test_processed.select_dtypes(include=[np.number]).columns

X_test_processed[num_features] = X_test_processed[num_features].clip(lower=0)

X_test_processed[num_features] = np.log1p(X_test_processed[num_features])

X_test_processed = X_test_processed.replace([np.inf, -np.inf], 0)

In [32]:
nan_columns = X_test_processed.isnull().sum()
nan_columns = nan_columns[nan_columns > 0]
if len(nan_columns) > 0:
    print("აღმოჩნდა null-ები შემდეგ სვეტებში: \n", nan_columns)

X_test_processed = X_test_processed.fillna(0)
 
log_predictions = loaded_model.predict(X_test_processed)

final_predictions = np.expm1(log_predictions)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("ფაილი 'submission.csv' წარმატებით შეიქმნა და მზადაა ასატვირთად!")

აღმოჩნდა null-ები შემდეგ სვეტებში: 
 GarageCars     1
KitchenQual    1
GarageArea     1
BsmtFinSF1     1
BsmtUnfSF      1
dtype: int64
ფაილი 'submission.csv' წარმატებით შეიქმნა და მზადაა ასატვირთად!
